In [1]:
import pandas as pd

message = pd.read_csv(
    'C:/Users/ASUS/Documents/nlp/spam.csv',
    sep='\t',
    names=['label', 'message'],
    encoding='latin-1'
)

print(message.head())

                                               label  message
0                                           v1,v2,,,      NaN
1  ham,"Go until jurong point, crazy.. Available ...      NaN
2               ham,Ok lar... Joking wif u oni...,,,      NaN
3  spam,Free entry in 2 a wkly comp to win FA Cup...      NaN
4  ham,U dun say so early hor... U c already then...      NaN


In [2]:

message['message']=message['label'].str.split(',').str[1]
message['label'] = message['label'].str.split(',').str[0]

In [3]:
message=message.drop(message.index[0])

In [4]:
message.head()

,label,message
1,ham,"""Go until jurong point"
2,ham,Ok lar... Joking wif u oni...
3,spam,Free entry in 2 a wkly comp to win FA Cup fina...
4,ham,U dun say so early hor... U c already then say...
5,ham,"""Nah I don't think he goes to usf"


In [20]:
message.isna().sum()

label      0
message    0
dtype: int64

In [5]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import re

import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
message.head(5)

,label,message
1,ham,"""Go until jurong point"
2,ham,Ok lar... Joking wif u oni...
3,spam,Free entry in 2 a wkly comp to win FA Cup fina...
4,ham,U dun say so early hor... U c already then say...
5,ham,"""Nah I don't think he goes to usf"


In [16]:

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

ps = PorterStemmer()

all_stopwords = set(stopwords.words('english'))

corpus = []

for i in range(0, len(message)):
    
    review = re.sub('[^a-zA-Z]', ' ', message['message'].iloc[i])
    
    review = review.lower()
    
    review = review.split()
    
    review = [ps.stem(word) for word in review if word not in all_stopwords]
    
    review = ' '.join(review)
    
    corpus.append(review)

In [24]:
y=pd.get_dummies(message['label'])
y=y.iloc[:,0].values 
y 

array([ True,  True, False, ...,  True,  True,  True], shape=(5574,))

In [25]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(corpus,y,test_size=0.20,random_state=0)

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=2500,ngram_range=(1,2))
X_train=tfidf.fit_transform(X_train).toarray()
X_test=tfidf.transform(X_test).toarray()


In [27]:

from sklearn.naive_bayes import MultinomialNB
spam_detect_model=MultinomialNB().fit(X_train,y_train)
y_pred=spam_detect_model.predict(X_test)
from sklearn.metrics import confusion_matrix
confusion_m=confusion_matrix(y_test,y_pred)
print(confusion_m)
from sklearn.metrics import accuracy_score
accuracy=accuracy_score(y_test,y_pred)
print(accuracy)
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))


[[127  38]
 [  1 949]]
0.9650224215246637
              precision    recall  f1-score   support

       False       0.99      0.77      0.87       165
        True       0.96      1.00      0.98       950

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115

